# Notebook 05 — Recommendations

**Project:** APAC Territory Planning  
**Objective:** Synthesise findings from notebooks 01-04 into prioritised, actionable recommendations for APAC sales leadership. Includes an executive memo and revenue impact summary.

**Business Questions Answered:**
1. Are APAC territories balanced by opportunity? → Partially — ARR Gini 0.339, load Gini 0.071
2. Which reps are over/under capacity? → India reps under-utilised, SEA/Greater China at 87-95%
3. Where is the whitespace? → 1,398 accounts, $521M ARR, 300 priority accounts identified
4. How should territories be realigned? → 3 new hires, overlay deployment, coverage model restructure
5. What is the revenue impact? → $398M priority whitespace ARR potential

In [ ]:
import pandas as pd
import duckdb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

In [ ]:
accounts      = pd.read_csv(f"{DATA_DIR}/accounts.csv")
reps          = pd.read_csv(f"{DATA_DIR}/reps.csv")
assignments   = pd.read_csv(f"{DATA_DIR}/assignments.csv")
opportunities = pd.read_csv(f"{DATA_DIR}/opportunities.csv")
customers     = pd.read_csv(f"{DATA_DIR}/customers.csv")
ws_scored     = pd.read_csv(f"{PROCESSED_DIR}/whitespace_scored.csv")

customers = customers[
    (pd.to_datetime(customers["contract_end"]) > pd.Timestamp("2025-03-01")) |
    (customers["renewal_flag"] == 1)
].copy()

print(f"Active customers: {len(customers):,}")

con = duckdb.connect()
con.register("accounts",      accounts)
con.register("reps",          reps)
con.register("assignments",   assignments)
con.register("opportunities", opportunities)
con.register("customers",     customers)
con.register("ws_scored",     ws_scored)

print("Tables registered:")
print(con.execute("SHOW TABLES").df().to_string(index=False))

## 1. APAC Territory Scorecard — Current State

In [ ]:
# Key metrics across all notebooks
scorecard = {
    "Total Accounts":              len(accounts),
    "Total Reps (excl. overlay)":  len(reps[reps["subregion"] != "Regional"]),
    "Assigned Accounts":           len(assignments[assignments["coverage_status"] == "Assigned"]),
    "Unassigned (Whitespace)":     len(assignments[assignments["coverage_status"] == "Unassigned"]),
    "Whitespace %":                f"{len(assignments[assignments['coverage_status'] == 'Unassigned']) / len(assignments) * 100:.1f}%",
    "Total Customers":             len(customers),
    "Customer Rate":               f"{len(customers) / len(accounts) * 100:.1f}%",
    "Total Customer ARR":          f"${customers['arr'].sum():,.0f}",
    "Avg ARR per Customer":        f"${customers['arr'].mean():,.0f}",
    "Total Whitespace Accounts":   len(ws_scored),
    "Priority Accounts (Tier 1)":  len(ws_scored[ws_scored["priority_flag"] == "Priority"]),
    "Priority Whitespace ARR":     f"${ws_scored[ws_scored['priority_flag'] == 'Priority']['estimated_arr'].sum():,.0f}",
    "Territory ARR Gini":          "0.339",
    "Territory Load Gini":         "0.071",
    "Open Pipeline":               f"${opportunities[~opportunities['stage'].isin(['Closed Won','Closed Lost'])]['arr_potential'].sum():,.0f}",
    "Overall Win Rate":            f"{len(opportunities[opportunities['stage'] == 'Closed Won']) / len(opportunities[opportunities['stage'].isin(['Closed Won','Closed Lost'])]) * 100:.1f}%",
}

print("APAC TERRITORY PLANNING — SCORECARD")
print("=" * 55)
for metric, value in scorecard.items():
    print(f"  {metric:<35} {str(value):>15}")
print("=" * 55)

## 2. Prioritised Recommendations

## Assumptions

Revenue impact estimates are based on the following assumptions:

| Assumption | Value | Source |
|---|---|---|
| Base conversion rate | 18.1% | Observed: active customers / assigned accounts |
| Avg ARR per customer | $35,369 | Observed: mean customer ARR in dataset |
| Tier 1 direct rep conversion | 26.5% | Base rate — highest intent accounts, direct ownership |
| Tier 2 direct rep conversion | 15.9% | 60% of base — less ready to buy than Tier 1 |
| BDR sequencing conversion | 14.6% | 55% of base — BDR qualifies, AE closes, longer cycle |
| India mixed conversion | 9.3% | 35% of base — BDR + marketing mix, SMB heavy market |
| Marketing nurture conversion | 7.9% | 30% of base — lowest intent, longest cycle |

**Caveats:**
- Conversion rate multipliers by coverage motion are based on B2B SaaS industry benchmarks
- Actual rates should be validated against historical CRM data before use in planning
- Revenue impact estimates assume ramp time is not factored in — first full year of productivity
- Subregion-level conversion rate variation is noted but not modelled — a recommended next step

In [ ]:
# Base conversion rate from observed data
base_conversion_rate = len(customers) / len(assignments[assignments["coverage_status"] == "Assigned"])
avg_arr = customers["arr"].mean()

# Tiered conversion rates by action type and tier
conversion_rates = {
    "direct_tier1":    base_conversion_rate,          # 26.5% — direct rep, Tier 1
    "direct_tier2":    base_conversion_rate * 0.60,   # 15.9% — direct rep, Tier 2
    "bdr":             base_conversion_rate * 0.55,   # 14.6% — BDR sequencing
    "marketing":       base_conversion_rate * 0.30,   # 7.9%  — marketing nurture
    "india_mixed":     base_conversion_rate * 0.35,   # 9.3%  — BDR + marketing mix
}

def estimate_impact(n_accounts, rate_key):
    return round(n_accounts * conversion_rates[rate_key] * avg_arr, -3)

# Account counts from whitespace analysis
gc_tier1     = len(ws_scored[(ws_scored["subregion"] == "Greater China") & (ws_scored["tier"] == "Tier 1")])
sea_tier1    = len(ws_scored[(ws_scored["subregion"] == "SEA") & (ws_scored["tier"] == "Tier 1")])
overlay_n    = 25
india_redir  = len(ws_scored[(ws_scored["subregion"] == "India") & (ws_scored["tier"].isin(["Tier 1", "Tier 2"]))])
gc_sea_tier2 = len(ws_scored[(ws_scored["subregion"].isin(["Greater China", "SEA"])) & (ws_scored["tier"] == "Tier 2")])
tier3_total  = len(ws_scored[ws_scored["tier"] == "Tier 3"])

recommendations = [
    {
        "Priority":       1,
        "Action":         "Hire 1 Enterprise rep in Greater China",
        "Rationale":      f"{gc_tier1} Tier 1 whitespace accounts, $149M ARR potential — largest single opportunity in APAC",
        "n_accounts":     gc_tier1,
        "rate_key":       "direct_tier1",
        "Timeframe":      "Q1 next FY",
        "Owner":          "APAC Sales Leader",
    },
    {
        "Priority":       2,
        "Action":         "Deploy overlay reps on top 25 Greater China + India Tier 1 accounts",
        "Rationale":      "Regional overlay reps at 0% utilisation — immediate coverage of highest value whitespace",
        "n_accounts":     overlay_n,
        "rate_key":       "direct_tier1",
        "Timeframe":      "Immediate",
        "Owner":          "Regional Overlay Reps",
    },
    {
        "Priority":       3,
        "Action":         "Hire 1 Mid-Market rep in SEA",
        "Rationale":      f"{sea_tier1} Tier 1 whitespace accounts, $134M ARR potential — Mid-Market segment underserved",
        "n_accounts":     sea_tier1,
        "rate_key":       "direct_tier1",
        "Timeframe":      "Q1 next FY",
        "Owner":          "APAC Sales Leader",
    },
    {
        "Priority":       4,
        "Action":         "Restructure India coverage model",
        "Rationale":      f"India reps at 54-62% load but lowest attainment — redirect {india_redir} Tier 1+2 accounts to BDR + marketing-led approach",
        "n_accounts":     india_redir,
        "rate_key":       "india_mixed",
        "Timeframe":      "Q2 next FY",
        "Owner":          "India Sales Manager",
    },
    {
        "Priority":       5,
        "Action":         "Launch Tier 2 BDR sequencing in Greater China and SEA",
        "Rationale":      f"{gc_sea_tier2} Tier 2 accounts sitting stale — structured outreach can convert without adding headcount",
        "n_accounts":     gc_sea_tier2,
        "rate_key":       "bdr",
        "Timeframe":      "Q2 next FY",
        "Owner":          "BDR Manager",
    },
    {
        "Priority":       6,
        "Action":         "Move Tier 3 accounts to marketing nurture",
        "Rationale":      f"{tier3_total} Tier 3 accounts with avg ARR $27K — nurture until they graduate to Tier 2",
        "n_accounts":     tier3_total,
        "rate_key":       "marketing",
        "Timeframe":      "Q2 next FY",
        "Owner":          "Marketing",
    },
    {
        "Priority":       7,
        "Action":         "Redeploy one SEA SMB rep to India",
        "Rationale":      f"SEA has 3 SMB reps with overlapping coverage — redeploying one to India addresses under-penetration without incremental headcount cost",
        "n_accounts":     int(len(ws_scored[(ws_scored["subregion"] == "India") & (ws_scored["tier"] == "Tier 2")])),
        "rate_key":       "india_mixed",
        "Timeframe":      "Q2 next FY",
        "Owner":          "APAC Sales Leader",
    },
]

recs_df = pd.DataFrame(recommendations)
recs_df["Revenue Impact"] = recs_df.apply(
    lambda x: estimate_impact(x["n_accounts"], x["rate_key"]), axis=1
)
total_impact = recs_df["Revenue Impact"].sum()

print("Base assumptions:")
print(f"  Observed conversion rate:      {base_conversion_rate:.1%}")
print(f"  Avg ARR per customer:          ${avg_arr:,.0f}")
print()
print("Conversion rates applied:")
for k, v in conversion_rates.items():
    print(f"  {k:<20} {v:.1%}")
print()
print("PRIORITISED RECOMMENDATIONS — APAC TERRITORY PLANNING")
print("=" * 100)
for _, row in recs_df.iterrows():
    print(f"\n  [{row['Priority']}] {row['Action']}")
    print(f"      Rationale:       {row['Rationale']}")
    print(f"      Accounts:        {row['n_accounts']:,}")
    print(f"      Conversion Rate: {conversion_rates[row['rate_key']]:.1%}")
    print(f"      Revenue Impact:  ${row['Revenue Impact']:,.0f}")
    print(f"      Timeframe:       {row['Timeframe']}")
    print(f"      Owner:           {row['Owner']}")
print()
print("=" * 100)
print(f"  Total Estimated Revenue Impact: ${total_impact:,.0f}")
print("=" * 100)

## 3. Revenue Impact by Recommendation

In [ ]:
fig = go.Figure()

def wrap_label(text, max_len=25):
    words = text.split()
    line1, line2 = [], []
    current_len = 0
    for word in words:
        if current_len + len(word) <= max_len:
            line1.append(word)
            current_len += len(word) + 1
        else:
            line2.append(word)
    return " ".join(line1) + "<br>" + " ".join(line2)

labels = [wrap_label(a) for a in recs_df["Action"]]

fig.add_trace(go.Bar(
    x=labels,
    y=recs_df["Revenue Impact"] / 1e6,
    marker_color=["#1565C0", "#1976D2", "#1E88E5", "#42A5F5", "#90CAF9", "#BBDEFB", "#E3F2FD"],
    text=[f"${v/1e6:.2f}M" for v in recs_df["Revenue Impact"]],
    textposition="outside"
))

max_val = recs_df["Revenue Impact"].max() / 1e6

fig.update_layout(
    title=f"Estimated Revenue Impact by Recommendation — Total ${total_impact/1e6:.1f}M",
    xaxis_title="Recommendation",
    yaxis=dict(title="Estimated ARR Impact (USD M)", range=[0, max_val * 1.25]),
    height=550,
    font=dict(size=12),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False
)

fig.show()
fig.write_image("../outputs/05_revenue_impact.png", width=1000, height=550, scale=2)
print("Saved: ../outputs/05_revenue_impact.png")

## 4. Executive Memo

### Summary

An analysis of 2,000 APAC accounts across 5 subregions and 18 sales reps reveals that while rep workload is well distributed (load Gini 0.071), revenue opportunity is not (ARR Gini 0.339). Less than 3.2% of total addressable market has been converted in any subregion. There are 1,398 unworked accounts representing $521M in whitespace ARR, of which 300 priority accounts alone account for $398M.

The core finding is that APAC is not a capacity problem — it is a coverage design problem. Reps are working hard in the wrong places.

---

### Key Findings

**1. Greater China is the largest untapped opportunity.**
122 Tier 1 Enterprise accounts with $149M ARR potential sit uncovered. With only 4 reps covering 480 accounts, Greater China is structurally under-resourced for its revenue potential. A single Enterprise hire focused on CN whitespace has the highest ROI of any recommended action.

**2. SEA is overloaded but converting well.**
SEA has the highest win rate (23.1%) and the most whitespace by volume (407 accounts). Reps are at 88-94% capacity with no room to prospect. A Mid-Market hire in SEA unlocks 111 Tier 1 accounts and $134M ARR potential.

**3. Regional overlay reps are underutilised.**
Both overlay reps currently have zero account assignments. Deploying them immediately on the top 25 Tier 1 accounts across Greater China and India requires no incremental headcount cost and generates an estimated $235K in near-term ARR.

**4. India needs a different playbook.**
India reps are at 54-62% capacity but have the lowest quota attainment in APAC. The market is SMB-heavy with avg ARR of $16K — direct rep coverage is not cost-effective. Redirecting India Tier 1+2 accounts to BDR sequencing and Tier 3 to marketing nurture better matches the market reality.

**5. Territory ARR is skewed toward SMB reps.**
SMB reps hold $48-77M in territory ARR while Enterprise reps hold only $11-13M. Enterprise accounts require deeper relationships and longer sales cycles — their territories should be enriched with higher-value accounts from the whitespace pool.

**6. SEA has redundant SMB coverage.**
SEA has 3 SMB reps covering 600 accounts at 88-91% load. Given the Mid-Market hire in Rec 3 will free up some SMB account load, redeploying 
one SEA SMB rep to India adds Enterprise-capable coverage to India's whitespace at zero incremental headcount cost.

---

### Recommended Actions

| Priority | Action | Impact | Timeframe |
|---|---|---|---|
| 1 | Hire 1 Enterprise rep — Greater China | $795K | Q1 next FY |
| 2 | Deploy overlay reps on top 25 Tier 1 accounts | $160K | Immediate |
| 3 | Hire 1 Mid-Market rep — SEA | $743K | Q1 next FY |
| 4 | Restructure India to BDR + marketing model | $269K | Q2 next FY |
| 5 | Launch Tier 2 BDR sequencing — Greater China + SEA | $1.1M | Q2 next FY |
| 6 | Move Tier 3 accounts to marketing nurture | $917K | Q2 next FY |
| 7 | Redeploy one SEA SMB rep to India | $220K | Q2 next FY |
| | **Total estimated revenue impact** | **$4.2M** | |

---

### Caveats and Next Steps

- Revenue impact estimates assume first full year of rep productivity with no ramp period factored in
- Conversion rate adjustments by coverage motion are based on B2B SaaS benchmarks — validate against historical CRM data before use in planning
- Territory tier thresholds are global — subregion-level clustering would produce more locally calibrated tiers
- ANZ win rate could not be calculated due to insufficient closed opportunity sample in the dataset

---

## Key Findings

1. **APAC is a coverage design problem, not a capacity problem:** Load Gini 0.071 (well balanced) but ARR Gini 0.339 (unequal opportunity) — reps are evenly utilised but unevenly endowed
2. **$4.2M revenue opportunity identified:** Across 7 prioritised actions spanning new hires, overlay deployment, BDR sequencing, marketing nurture and rep redeployment
3. **Greatest ROI is in Greater China:** 1 Enterprise hire unlocks $795K — highest single action return
4. **Overlay reps are the fastest win:** Zero incremental cost, $160K estimated impact, deployable immediately
5. **India requires a fundamentally different go-to-market:** Direct rep coverage is not cost-effective in an SMB-heavy market with $16K avg ARR
6. **SEA SMB redeployment to India:** Zero headcount cost, $220K impact — highest efficiency recommendation in the plan
7. **SMB reps are carrying disproportionate ARR weight:** Territory rebalancing should enrich Enterprise rep portfolios with higher-value whitespace accounts — 234 specific accounts flagged in rebalancing_list.csv
8. **Total whitespace ARR of $521M remains untouched:** Even capturing 1% more of TAM across APAC would materially move revenue — the opportunity cost of inaction is high
9. **Active customer filter applied:** 131 churned/expired customers excluded — all metrics reflect current active revenue only